In [ ]:
# Entity Extraction Method Comparison
# This script compares two different entity extraction methods against a ground truth dataset

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Set
from collections import Counter
from fuzzywuzzy import fuzz
from matplotlib_venn import venn3, venn2
from IPython.display import display, HTML

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12})
sns.set_palette("colorblind")

# File paths - update these to your actual file paths
ground_truth_file = "data/dataset/mine_hpo.json"
method1_file = "data/results/agents/hpo/extracted_multi-hybrid_iter2_temp5.json" # multi 
method2_file = "data/results/agents/hpo/extract_t/extracted_t001_i1_r1.json" # single iter

# Labels for methods
method1_name = "Multi"  # Update with meaningful names
method2_name = "Single"
# method1_file = "data/results/agents/hpo/extract_t/extracted_t001_i2_r1.json" # single iter
# method1_name = "double"


# Load data
with open(ground_truth_file, 'r') as f:
    ground_truth_data = json.load(f)

with open(method1_file, 'r') as f:
    method1_data = json.load(f)
    
with open(method2_file, 'r') as f:
    method2_data = json.load(f)

print(f"Loaded ground truth data with {len(ground_truth_data)} cases")
print(f"Loaded {method1_name} data with {len(method1_data)} cases")
print(f"Loaded {method2_name} data with {len(method2_data)} cases")

def normalize_text(text: str) -> str:
    """Normalize text by converting to lowercase and removing extra whitespace."""
    return ' '.join(text.lower().split())

def extract_ground_truth_entities(ground_truth_data: Dict) -> Dict[str, List[str]]:
    """Extract ground truth entities."""
    result = {}
    
    if isinstance(ground_truth_data, dict):
        for case_id, case_data in ground_truth_data.items():
            case_id_str = str(case_id)
            entities = []
            
            # Try different possible field names
            for field in ["phenotypes", "ground_truth", "hpo_terms", "entities", "ground_truth_entities"]:
                if field in case_data:
                    field_data = case_data[field]
                    if isinstance(field_data, list):
                        for item in field_data:
                            if isinstance(item, dict):
                                # Try different field names for the entity
                                for entity_field in ["phenotype", "phenotype_name", "name", "text", "entity"]:
                                    if entity_field in item:
                                        entities.append(item[entity_field])
                                        break
                            elif isinstance(item, str):
                                entities.append(item)
                    break
            
            # Add non-empty lists to result
            if entities:
                result[case_id_str] = entities
    
    return result

def extract_entities(entities_data: Dict) -> Dict[str, List[str]]:
    """Extract entities from extraction method results."""
    result = {}
    
    if isinstance(entities_data, dict):
        for case_id, case_data in entities_data.items():
            case_id_str = str(case_id)
            entities = []
            
            # For step1 output format (entities_with_contexts)
            if isinstance(case_data, dict) and "entities_with_contexts" in case_data:
                for entity_data in case_data["entities_with_contexts"]:
                    if isinstance(entity_data, dict) and "entity" in entity_data:
                        entities.append(entity_data["entity"])
            
            # For step2 output format (verified_phenotypes)
            elif isinstance(case_data, dict) and "verified_phenotypes" in case_data:
                for phenotype_data in case_data["verified_phenotypes"]:
                    if isinstance(phenotype_data, dict):
                        if "original_entity" in phenotype_data:
                            entities.append(phenotype_data["original_entity"])
                        elif "entity" in phenotype_data:
                            entities.append(phenotype_data["entity"])
                        elif "phenotype" in phenotype_data:
                            entities.append(phenotype_data["phenotype"])
            
            # Add non-empty lists to result
            if entities:
                result[case_id_str] = entities
    
    return result

# Extract entities
ground_truth_entities = extract_ground_truth_entities(ground_truth_data)
method1_entities = extract_entities(method1_data)
method2_entities = extract_entities(method2_data)

print(f"Extracted entities from {len(ground_truth_entities)} ground truth cases")
print(f"Extracted entities from {len(method1_entities)} {method1_name} cases")
print(f"Extracted entities from {len(method2_entities)} {method2_name} cases")

def calculate_similarity_matrix(predictions: List[str], ground_truth: List[str], 
                              similarity_threshold: float = 50.0) -> np.ndarray:
    """Calculate similarity matrix between predictions and ground truth."""
    similarity_matrix = np.zeros((len(predictions), len(ground_truth)))
    
    for i, pred in enumerate(predictions):
        for j, truth in enumerate(ground_truth):
            similarity = fuzz.ratio(normalize_text(pred), normalize_text(truth))
            if similarity >= similarity_threshold:
                similarity_matrix[i, j] = similarity
    
    return similarity_matrix

def find_best_matches(similarity_matrix: np.ndarray) -> List[Tuple[int, int, float]]:
    """Find best matches using a greedy approach."""
    matches = []
    sim_matrix = similarity_matrix.copy()
    
    while np.max(sim_matrix) > 0:
        max_val = np.max(sim_matrix)
        max_pos = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
        
        pred_idx, truth_idx = max_pos
        matches.append((pred_idx, truth_idx, max_val))
        
        # Mark row and column as used
        sim_matrix[pred_idx, :] = 0
        sim_matrix[:, truth_idx] = 0
    
    return matches

def evaluate_entities(predictions: List[str], ground_truth: List[str], 
                     similarity_threshold: float = 50.0) -> Dict:
    """Evaluate predictions against ground truth with fuzzy matching."""
    # First, deduplicate both lists
    unique_predictions = list(set(predictions))
    unique_ground_truth = list(set(ground_truth))
    
    # Store original counts
    pred_counter = Counter(predictions)
    truth_counter = Counter(ground_truth)
    
    # Default result structure
    result = {
        "precision": 0.0,
        "recall": 0.0,
        "f1_score": 0.0,
        "tp_count": 0,
        "fp_count": 0,
        "fn_count": 0,
        "true_positives": [],
        "false_positives": [],
        "false_negatives": []
    }
    
    # Handle empty sets
    if not unique_predictions or not unique_ground_truth:
        if not unique_predictions:
            result["precision"] = 1.0
        result["false_positives"] = unique_predictions
        result["false_negatives"] = unique_ground_truth
        result["fp_count"] = len(unique_predictions)
        result["fn_count"] = len(unique_ground_truth)
        return result
    
    # Calculate similarity matrix
    similarity_matrix = calculate_similarity_matrix(
        unique_predictions, unique_ground_truth, similarity_threshold
    )
    
    # Find best matches
    matches = find_best_matches(similarity_matrix)
    
    # Process matches
    matched_pred_indices = set()
    matched_truth_indices = set()
    
    for pred_idx, truth_idx, similarity in matches:
        pred_text = unique_predictions[pred_idx]
        truth_text = unique_ground_truth[truth_idx]
        
        result["true_positives"].append({
            "prediction": pred_text,
            "ground_truth": truth_text,
            "similarity": similarity
        })
        
        matched_pred_indices.add(pred_idx)
        matched_truth_indices.add(truth_idx)
    
    # Identify false positives and negatives
    false_positive_indices = set(range(len(unique_predictions))) - matched_pred_indices
    false_negative_indices = set(range(len(unique_ground_truth))) - matched_truth_indices
    
    result["false_positives"] = [unique_predictions[i] for i in false_positive_indices]
    result["false_negatives"] = [unique_ground_truth[i] for i in false_negative_indices]
    
    # Calculate metrics
    result["tp_count"] = len(matched_pred_indices)
    result["fp_count"] = len(false_positive_indices)
    result["fn_count"] = len(false_negative_indices)
    
    # Calculate weighted true positives based on similarity scores
    weighted_tp = sum(similarity / 100.0 for _, _, similarity in matches)
    
    # Calculate precision and recall
    if unique_predictions:
        result["precision"] = weighted_tp / len(unique_predictions)
    if unique_ground_truth:
        result["recall"] = weighted_tp / len(unique_ground_truth)
    
    # Calculate F1 score
    if result["precision"] + result["recall"] > 0:
        result["f1_score"] = 2 * (result["precision"] * result["recall"]) / (result["precision"] + result["recall"])
    
    return result

def evaluate_all_cases(method_entities: Dict[str, List[str]], ground_truth_entities: Dict[str, List[str]],
                      similarity_threshold: float = 50.0) -> Dict[str, Dict]:
    """Evaluate all cases for a method."""
    results = {}
    
    # Get all unique case IDs
    all_case_ids = set(list(method_entities.keys()) + list(ground_truth_entities.keys()))
    
    for case_id in all_case_ids:
        # Get entities for this case
        method_case_entities = method_entities.get(case_id, [])
        ground_truth_case_entities = ground_truth_entities.get(case_id, [])
        
        # Skip if both are empty
        if not method_case_entities and not ground_truth_case_entities:
            continue
        
        # Evaluate this case
        results[case_id] = evaluate_entities(
            method_case_entities, ground_truth_case_entities, similarity_threshold
        )
    
    return results

# Set similarity threshold
similarity_threshold = 50.0

# Evaluate each method
method1_results = evaluate_all_cases(method1_entities, ground_truth_entities, similarity_threshold)
method2_results = evaluate_all_cases(method2_entities, ground_truth_entities, similarity_threshold)

print(f"Evaluated {len(method1_results)} cases for {method1_name}")
print(f"Evaluated {len(method2_results)} cases for {method2_name}")

def calculate_overall_metrics(results: Dict[str, Dict]) -> Dict:
    """Calculate overall metrics across all cases."""
    # Initialize counters
    total_tp = 0
    total_fp = 0
    total_fn = 0
    weighted_tp_sum = 0
    
    # Aggregate counts
    for case_id, case_result in results.items():
        total_tp += case_result["tp_count"]
        total_fp += case_result["fp_count"]
        total_fn += case_result["fn_count"]
        
        # Calculate weighted true positives
        for tp in case_result["true_positives"]:
            weighted_tp_sum += tp["similarity"] / 100.0
    
    # Calculate metrics
    precision = weighted_tp_sum / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = weighted_tp_sum / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "tp_count": total_tp,
        "fp_count": total_fp,
        "fn_count": total_fn,
        "weighted_tp": weighted_tp_sum
    }

# Calculate overall metrics
method1_overall = calculate_overall_metrics(method1_results)
method2_overall = calculate_overall_metrics(method2_results)

# Create a comparison DataFrame
overall_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1 Score", "True Positives", "False Positives", "False Negatives"],
    method1_name: [
        method1_overall["precision"],
        method1_overall["recall"],
        method1_overall["f1_score"],
        method1_overall["tp_count"],
        method1_overall["fp_count"],
        method1_overall["fn_count"]
    ],
    method2_name: [
        method2_overall["precision"],
        method2_overall["recall"],
        method2_overall["f1_score"],
        method2_overall["tp_count"],
        method2_overall["fp_count"],
        method2_overall["fn_count"]
    ]
})

# Display overall metrics
print("\nOverall Metrics Comparison:")
print(overall_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# Plot comparison of precision, recall, and F1 score
metrics_to_plot = overall_df.iloc[:3].copy()
metrics_to_plot = pd.melt(metrics_to_plot, id_vars=["Metric"], var_name="Method", value_name="Score")

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x="Metric", y="Score", hue="Method", data=metrics_to_plot)
plt.title("Overall Performance Comparison")
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on bars
for p in bar_plot.patches:
    bar_plot.annotate(f"{p.get_height():.3f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha = 'center', va = 'bottom', xytext = (0, 5),
                     textcoords = 'offset points')

plt.tight_layout()
plt.show()

def identify_diverging_cases(method1_results: Dict[str, Dict], method2_results: Dict[str, Dict]) -> pd.DataFrame:
    """Identify cases where methods perform differently."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_entities.keys())
    
    # Calculate performance difference for each case
    case_data = []
    for case_id in common_cases:
        # Skip cases without ground truth
        if not ground_truth_entities.get(case_id, []):
            continue
            
        m1_result = method1_results[case_id]
        m2_result = method2_results[case_id]
        
        # Calculate absolute differences
        f1_diff = abs(m1_result["f1_score"] - m2_result["f1_score"])
        precision_diff = abs(m1_result["precision"] - m2_result["precision"])
        recall_diff = abs(m1_result["recall"] - m2_result["recall"])
        
        # Determine which method is better for this case
        better_method = method1_name if m1_result["f1_score"] > m2_result["f1_score"] else method2_name
        
        # Count of entities
        ground_truth_count = len(ground_truth_entities[case_id])
        m1_count = len(method1_entities.get(case_id, []))
        m2_count = len(method2_entities.get(case_id, []))
        
        case_data.append({
            "case_id": case_id,
            f"{method1_name} F1": m1_result["f1_score"],
            f"{method2_name} F1": m2_result["f1_score"],
            "F1 Difference": f1_diff,
            f"{method1_name} Precision": m1_result["precision"],
            f"{method2_name} Precision": m2_result["precision"],
            "Precision Difference": precision_diff,
            f"{method1_name} Recall": m1_result["recall"],
            f"{method2_name} Recall": m2_result["recall"],
            "Recall Difference": recall_diff,
            "Better Method": better_method,
            "Ground Truth Count": ground_truth_count,
            f"{method1_name} Count": m1_count,
            f"{method2_name} Count": m2_count,
        })
    
    # Create DataFrame and sort by F1 difference
    divergence_df = pd.DataFrame(case_data)
    divergence_df = divergence_df.sort_values("F1 Difference", ascending=False)
    
    return divergence_df

# Identify diverging cases
divergence_df = identify_diverging_cases(method1_results, method2_results)

# Display top 10 diverging cases
print("\nTop 10 Cases with Largest Performance Differences:")
print(divergence_df.head(10).to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# Plot F1 score differences for top 10 cases
top_cases = divergence_df.head(10)

plt.figure(figsize=(12, 6))
ind = np.arange(len(top_cases))
width = 0.35

bars1 = plt.bar(ind - width/2, top_cases[f"{method1_name} F1"], width, label=method1_name)
bars2 = plt.bar(ind + width/2, top_cases[f"{method2_name} F1"], width, label=method2_name)

plt.ylabel('F1 Score')
plt.title('F1 Scores for Top Diverging Cases')
plt.xticks(ind, top_cases['case_id'], rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

def analyze_entity_sets(case_id: str) -> Dict:
    """Analyze entity sets for a specific case."""
    # Get entities for this case
    ground_truth = set(ground_truth_entities.get(case_id, []))
    method1_set = set(method1_entities.get(case_id, []))
    method2_set = set(method2_entities.get(case_id, []))
    
    # Get true positives for each method (using fuzzy matching results)
    if case_id in method1_results:
        method1_tps = {tp["prediction"] for tp in method1_results[case_id]["true_positives"]}
    else:
        method1_tps = set()
        
    if case_id in method2_results:
        method2_tps = {tp["prediction"] for tp in method2_results[case_id]["true_positives"]}
    else:
        method2_tps = set()
    
    # Analyze set differences
    return {
        "case_id": case_id,
        "ground_truth": ground_truth,
        f"{method1_name}_entities": method1_set,
        f"{method2_name}_entities": method2_set,
        f"{method1_name}_only_correct": method1_tps - method2_tps,  # Correct in method1 but not method2
        f"{method2_name}_only_correct": method2_tps - method1_tps,  # Correct in method2 but not method1
        "both_correct": method1_tps & method2_tps,  # Correct in both methods
        "missed_by_both": ground_truth - {gt for tp in method1_results.get(case_id, {}).get("true_positives", []) 
                                       for gt in [tp.get("ground_truth")]} -
                          {gt for tp in method2_results.get(case_id, {}).get("true_positives", []) 
                                       for gt in [tp.get("ground_truth")]},  # Missed by both methods
    }

# Analyze top 5 diverging cases
top_5_case_ids = divergence_df.head(5)["case_id"].tolist()
top_5_analyses = [analyze_entity_sets(case_id) for case_id in top_5_case_ids]

# Print analysis for each case
for i, analysis in enumerate(top_5_analyses):
    case_id = analysis["case_id"]
    
    print(f"\n==== Case {case_id} (Rank {i+1}) ====")
    
    # Get F1 scores for this case
    method1_f1 = method1_results.get(case_id, {}).get("f1_score", 0)
    method2_f1 = method2_results.get(case_id, {}).get("f1_score", 0)
    
    # Display scores
    print(f"{method1_name} F1: {method1_f1:.4f}, {method2_name} F1: {method2_f1:.4f}, Difference: {abs(method1_f1 - method2_f1):.4f}")
    
    # Create sets for Venn diagram
    set1 = analysis[f"{method1_name}_entities"]
    set2 = analysis[f"{method2_name}_entities"]
    set3 = analysis["ground_truth"]
    
    # Plot Venn diagram
    plt.figure(figsize=(10, 6))
    venn3([set1, set2, set3], 
          (method1_name, method2_name, "Ground Truth"))
    plt.title(f"Entity Set Comparison for Case {case_id}")
    plt.tight_layout()
    plt.show()
    
    # Display entity details
    print(f"\nEntity Details for Case {case_id}:")
    
    print(f"\nEntities correct only in {method1_name}:")
    for entity in analysis[f"{method1_name}_only_correct"]:
        print(f"  - {entity}")
    
    print(f"\nEntities correct only in {method2_name}:")
    for entity in analysis[f"{method2_name}_only_correct"]:
        print(f"  - {entity}")
    
    print("\nEntities correct in both methods:")
    for entity in analysis["both_correct"]:
        print(f"  - {entity}")
    
    print("\nEntities missed by both methods:")
    for entity in analysis["missed_by_both"]:
        print(f"  - {entity}")
    
    # Show original text if available
    if case_id in method1_data and "clinical_text" in method1_data[case_id]:
        clinical_text = method1_data[case_id]["clinical_text"]
        print(f"\nOriginal Clinical Text (truncated):")
        print(clinical_text[:500] + "..." if len(clinical_text) > 500 else clinical_text)
    elif case_id in method2_data and "clinical_text" in method2_data[case_id]:
        clinical_text = method2_data[case_id]["clinical_text"]
        print(f"\nOriginal Clinical Text (truncated):")
        print(clinical_text[:500] + "..." if len(clinical_text) > 500 else clinical_text)
    
    print("\n" + "-"*80)

def analyze_method_strengths(method1_results, method2_results):
    """Analyze strengths and weaknesses of each method across all cases."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys())
    
    # Initialize sets for strengths analysis
    method1_only_correct = set()
    method2_only_correct = set()
    both_correct = set()
    missed_by_both = set()
    
    # Collect entities across all cases
    for case_id in common_cases:
        if case_id not in ground_truth_entities:
            continue
            
        # Get true positives for each method
        method1_tps = {tp["prediction"] for tp in method1_results[case_id]["true_positives"]}
        method2_tps = {tp["prediction"] for tp in method2_results[case_id]["true_positives"]}
        
        # Update sets
        method1_only_correct.update(method1_tps - method2_tps)
        method2_only_correct.update(method2_tps - method1_tps)
        both_correct.update(method1_tps & method2_tps)
        
        # Get ground truth items missed by both methods
        method1_gt_matches = {tp["ground_truth"] for tp in method1_results[case_id]["true_positives"]}
        method2_gt_matches = {tp["ground_truth"] for tp in method2_results[case_id]["true_positives"]}
        missed = set(ground_truth_entities[case_id]) - method1_gt_matches - method2_gt_matches
        missed_by_both.update(missed)
    
    # Count method performances by case
    method1_better_count = 0
    method2_better_count = 0
    tie_count = 0
    
    for case_id in common_cases:
        if case_id not in ground_truth_entities:
            continue
            
        m1_f1 = method1_results[case_id]["f1_score"]
        m2_f1 = method2_results[case_id]["f1_score"]
        
        if abs(m1_f1 - m2_f1) < 0.01:  # Consider as tie if difference is very small
            tie_count += 1
        elif m1_f1 > m2_f1:
            method1_better_count += 1
        else:
            method2_better_count += 1
    
    # Create summary DataFrame
    summary_df = pd.DataFrame({
        "Category": [f"Entities found only by {method1_name}", 
                    f"Entities found only by {method2_name}",
                    "Entities found by both methods",
                    "Entities missed by both methods",
                    f"Cases where {method1_name} performed better",
                    f"Cases where {method2_name} performed better",
                    "Cases with similar performance"],
        "Count": [len(method1_only_correct),
                len(method2_only_correct),
                len(both_correct),
                len(missed_by_both),
                method1_better_count,
                method2_better_count,
                tie_count]
    })
    
    # Return sets and summary
    return {
        "summary": summary_df,
        "method1_only": method1_only_correct,
        "method2_only": method2_only_correct,
        "both": both_correct,
        "missed": missed_by_both
    }

# Analyze strengths
strengths_analysis = analyze_method_strengths(method1_results, method2_results)

# Display summary
print("\nOverall Method Comparison:")
print(strengths_analysis["summary"].to_string(index=False))

# Plot summary
plt.figure(figsize=(12, 6))
entity_data = strengths_analysis["summary"].iloc[:4].copy()
sns.barplot(x="Category", y="Count", data=entity_data)
plt.title("Entity Detection Comparison")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
case_data = strengths_analysis["summary"].iloc[4:].copy()
sns.barplot(x="Category", y="Count", data=case_data)
plt.title("Case Performance Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Display most common unique entities found by each method
print("\nMost Common Entities Found Only by Each Method:")

# Get most common entities unique to each method
method1_common = Counter(list(strengths_analysis["method1_only"])).most_common(15)
method2_common = Counter(list(strengths_analysis["method2_only"])).most_common(15)

# Create DataFrames for display
method1_common_df = pd.DataFrame(method1_common, columns=[f"Entity unique to {method1_name}", "Count"])
method2_common_df = pd.DataFrame(method2_common, columns=[f"Entity unique to {method2_name}", "Count"])

# Display side by side
print(pd.concat([method1_common_df, method2_common_df], axis=1).to_string(index=False))

print("""
Conclusion:

Based on the analysis above, we can draw the following conclusions about the two entity extraction methods:

1. Overall Performance:
   - [Describe which method performs better overall and why]

2. Strengths of Method A:
   - [List specific types of entities or cases where Method A excels]

3. Strengths of Method B:
   - [List specific types of entities or cases where Method B excels]

4. Opportunities for Improvement:
   - [Describe entities missed by both methods and potential areas for improvement]

5. Recommendations:
   - [Based on the analysis, provide recommendations for when to use each method or how to combine them]

This analysis provides insights into how the two extraction methods compare and can guide decisions about which method to use in different scenarios or how to potentially combine their strengths.
""")

def analyze_missed_entities(method1_results, method2_results):
    """Analyze entities missed by both methods across all cases."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_entities.keys())
    
    # Initialize collections for missed entities
    all_missed = []
    missed_by_case = {}
    
    # Analyze missed entities for each case
    for case_id in common_cases:
        # Get ground truth entities
        gt_entities = ground_truth_entities.get(case_id, [])
        if not gt_entities:
            continue
            
        # Get matched ground truth entities for each method
        method1_gt_matches = {tp["ground_truth"] for tp in method1_results[case_id]["true_positives"]}
        method2_gt_matches = {tp["ground_truth"] for tp in method2_results[case_id]["true_positives"]}
        
        # Find entities missed by both methods
        missed = set(gt_entities) - method1_gt_matches - method2_gt_matches
        
        # Add to collections
        if missed:
            missed_by_case[case_id] = list(missed)
            all_missed.extend(missed)
    
    # Count occurrences of each missed entity
    missed_counts = Counter(all_missed)
    
    return {
        "missed_by_case": missed_by_case,
        "most_common_missed": missed_counts.most_common(30),  # Show top 30 most commonly missed
        "total_missed_unique": len(missed_counts),
        "total_missed_occurrences": sum(missed_counts.values())
    }

# Run the missed entities analysis
print("\n===== DETAILED ANALYSIS OF MISSED ENTITIES =====")
missed_analysis = analyze_missed_entities(method1_results, method2_results)

# Display overall statistics
print(f"\nTotal unique entities missed by both methods: {missed_analysis['total_missed_unique']}")
print(f"Total occurrences of missed entities: {missed_analysis['total_missed_occurrences']}")

# Display most commonly missed entities
print("\nTop 30 Most Commonly Missed Entities:")
for entity, count in missed_analysis['most_common_missed']:
    print(f"  - '{entity}' (missed in {count} cases)")

# Display cases with the most missed entities
cases_by_missed_count = sorted(
    [(case_id, len(entities)) for case_id, entities in missed_analysis['missed_by_case'].items()],
    key=lambda x: x[1], reverse=True
)

print("\nTop 10 Cases with Most Missed Entities:")
for i, (case_id, count) in enumerate(cases_by_missed_count[:10]):
    print(f"\nCase {case_id}: {count} missed entities")
    
    # Show all missed entities for these top cases
    missed_entities = missed_analysis['missed_by_case'][case_id]
    for entity in missed_entities:
        print(f"  - '{entity}'")
    
    # Show original text if available
    if case_id in method1_data and "clinical_text" in method1_data[case_id]:
        clinical_text = method1_data[case_id]["clinical_text"]
        print(f"\nExcerpt from Clinical Text:")
        print(clinical_text[:300] + "..." if len(clinical_text) > 300 else clinical_text)
    elif case_id in method2_data and "clinical_text" in method2_data[case_id]:
        clinical_text = method2_data[case_id]["clinical_text"]
        print(f"\nExcerpt from Clinical Text:")
        print(clinical_text[:300] + "..." if len(clinical_text) > 300 else clinical_text)
    
    print("-"*80)

# Optionally, explore patterns in missed entities
print("\nAnalyzing patterns in missed entities...")

# Count word lengths of missed entities
word_lengths = [len(entity.split()) for entity in all_missed]
avg_length = sum(word_lengths) / len(word_lengths) if word_lengths else 0

print(f"Average word count in missed entities: {avg_length:.2f}")
print(f"Single-word entities missed: {word_lengths.count(1)} ({word_lengths.count(1)/len(word_lengths)*100:.1f}%)")
print(f"Multi-word entities missed: {len(word_lengths) - word_lengths.count(1)} ({(len(word_lengths) - word_lengths.count(1))/len(word_lengths)*100:.1f}%)")

# You could also add visualization of missed entities
plt.figure(figsize=(10, 6))
plt.hist(word_lengths, bins=range(1, max(word_lengths) + 2), alpha=0.7)
plt.title("Distribution of Word Counts in Missed Entities")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.xticks(range(1, max(word_lengths) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Verifier Comparisons

In [ ]:
# Verifier Performance Comparison
# This script compares two different verification methods against a ground truth dataset


# Verifier Performance Comparison
# This script compares two different verification methods against a ground truth dataset


import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Set
from collections import Counter
from fuzzywuzzy import fuzz
from matplotlib_venn import venn3, venn2
from IPython.display import display, HTML

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12})
sns.set_palette("colorblind")

# File paths - update these to your actual file paths
ground_truth_file = "data/dataset/mine_hpo.json"
method1_file = "data/results/agents/hpo/val_ext_i1_t001_t01_v2_v2.json" # multi 
method2_file = "data/results/agents/hpo/val_ext_i1_t001_t01_v3_v3.json" # single iter

# Labels for methods
method1_name = "Claude"  # Update with meaningful names
method2_name = "Direct"


method1_file = "data/results/agents/hpo/val_ext_i1_t001_t01.json" # multi 
method1_name = "Original"

method1_file = 'data/results/agents/hpo/multi_t5_v3_v3.json'
method1_name = "Multi + Direct"
# Verifier Performance Comparison
# This script compares two different verification methods against a ground truth dataset
# Load data
with open(ground_truth_file, 'r') as f:
    ground_truth_data = json.load(f)

with open(method1_file, 'r') as f:
    method1_data = json.load(f)
    
with open(method2_file, 'r') as f:
    method2_data = json.load(f)

print(f"Loaded ground truth data with {len(ground_truth_data)} cases")
print(f"Loaded {method1_name} data with {len(method1_data)} cases")
print(f"Loaded {method2_name} data with {len(method2_data)} cases")

def normalize_text(text: str) -> str:
    """Normalize text by converting to lowercase and removing extra whitespace."""
    return ' '.join(text.lower().split())

def extract_ground_truth_phenotypes(ground_truth_data: Dict) -> Dict[str, List[str]]:
    """Extract ground truth phenotypes."""
    result = {}
    
    if isinstance(ground_truth_data, dict):
        for case_id, case_data in ground_truth_data.items():
            case_id_str = str(case_id)
            phenotypes = []
            
            # Try different possible field names
            for field in ["phenotypes", "ground_truth", "hpo_terms", "entities", "ground_truth_entities"]:
                if field in case_data:
                    field_data = case_data[field]
                    if isinstance(field_data, list):
                        for item in field_data:
                            if isinstance(item, dict):
                                # Try different field names for the phenotype
                                for entity_field in ["phenotype", "phenotype_name", "name", "text", "entity"]:
                                    if entity_field in item:
                                        phenotypes.append(item[entity_field])
                                        break
                            elif isinstance(item, str):
                                phenotypes.append(item)
                    break
            
            # Add non-empty lists to result
            if phenotypes:
                result[case_id_str] = phenotypes
    
    return result

def extract_verified_phenotypes(verifier_data: Dict) -> Dict[str, List[str]]:
    """
    Extract verified phenotypes from step2_verify output format.
    
    The verifier outputs have this structure:
    {
        case_id: {
            "verified_phenotypes": [
                {
                    "status": "direct_phenotype" or "implied_phenotype",
                    "phenotype": phenotype_value,
                    "original_entity": original_entity_value,
                    "confidence": confidence_score,
                    "method": method_name
                },
                ...
            ],
            "stats": {...},
            "original_text": "..."
        },
        ...
    }
    
    Args:
        verifier_data (Dict): Verification results dictionary
    
    Returns:
        Dict[str, List[str]]: Dictionary mapping case IDs to lists of phenotypes
    """
    result = {}
    
    # Ensure input is a dictionary
    if not isinstance(verifier_data, dict):
        return result
    
    if "results" in verifier_data:
        verifier_data = verifier_data["results"]
    # Iterate through each case
    for case_id, case_data in verifier_data.items():
        # Convert case_id to string to ensure consistent key type
        case_id_str = str(case_id)
        # print(verifier_data)
        
        # Check if verified_phenotypes exist and is a list
        if not isinstance(case_data, dict) or 'verified_phenotypes' not in case_data:
            continue
        
        verified_list = case_data['verified_phenotypes']
        
        # Only process if verified_phenotypes is a list
        if not isinstance(verified_list, list):
            continue
        
        # Extract phenotypes
        phenotypes = []
        for item in verified_list:
            # Ensure item is a dictionary
            if not isinstance(item, dict):
                continue
            
            # Extract phenotype, prioritizing 'phenotype' field
            if 'phenotype' in item:
                phenotypes.append(item['phenotype'])
            elif 'original_entity' in item:
                phenotypes.append(item['original_entity'])
        
        # Add non-empty lists to result
        if phenotypes:
            result[case_id_str] = phenotypes
        # print(result)
    return result

# Extract phenotypes
ground_truth_phenotypes = extract_ground_truth_phenotypes(ground_truth_data)
method1_phenotypes = extract_verified_phenotypes(method1_data)
method2_phenotypes = extract_verified_phenotypes(method2_data)

print(f"Extracted phenotypes from {len(ground_truth_phenotypes)} ground truth cases")
print(f"Extracted verified phenotypes from {len(method1_phenotypes)} {method1_name} cases")
print(f"Extracted verified phenotypes from {len(method2_phenotypes)} {method2_name} cases")

def calculate_similarity_matrix(predictions: List[str], ground_truth: List[str], 
                              similarity_threshold: float = 50.0) -> np.ndarray:
    """Calculate similarity matrix between predictions and ground truth."""
    similarity_matrix = np.zeros((len(predictions), len(ground_truth)))
    
    for i, pred in enumerate(predictions):
        for j, truth in enumerate(ground_truth):
            similarity = fuzz.ratio(normalize_text(pred), normalize_text(truth))
            if similarity >= similarity_threshold:
                similarity_matrix[i, j] = similarity
    
    return similarity_matrix

def find_best_matches(similarity_matrix: np.ndarray) -> List[Tuple[int, int, float]]:
    """Find best matches using a greedy approach."""
    matches = []
    sim_matrix = similarity_matrix.copy()
    
    while np.max(sim_matrix) > 0:
        max_val = np.max(sim_matrix)
        max_pos = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
        
        pred_idx, truth_idx = max_pos
        matches.append((pred_idx, truth_idx, max_val))
        
        # Mark row and column as used
        sim_matrix[pred_idx, :] = 0
        sim_matrix[:, truth_idx] = 0
    
    return matches

def evaluate_phenotypes(predictions: List[str], ground_truth: List[str], 
                     similarity_threshold: float = 50.0) -> Dict:
    """Evaluate predictions against ground truth with fuzzy matching."""
    # First, deduplicate both lists
    unique_predictions = list(set(predictions))
    unique_ground_truth = list(set(ground_truth))
    
    # Store original counts
    pred_counter = Counter(predictions)
    truth_counter = Counter(ground_truth)
    
    # Default result structure
    result = {
        "precision": 0.0,
        "recall": 0.0,
        "f1_score": 0.0,
        "tp_count": 0,
        "fp_count": 0,
        "fn_count": 0,
        "true_positives": [],
        "false_positives": [],
        "false_negatives": []
    }
    
    # Handle empty sets
    if not unique_predictions or not unique_ground_truth:
        if not unique_predictions:
            result["precision"] = 1.0
        result["false_positives"] = unique_predictions
        result["false_negatives"] = unique_ground_truth
        result["fp_count"] = len(unique_predictions)
        result["fn_count"] = len(unique_ground_truth)
        return result
    
    # Calculate similarity matrix
    similarity_matrix = calculate_similarity_matrix(
        unique_predictions, unique_ground_truth, similarity_threshold
    )
    
    # Find best matches
    matches = find_best_matches(similarity_matrix)
    
    # Process matches
    matched_pred_indices = set()
    matched_truth_indices = set()
    
    for pred_idx, truth_idx, similarity in matches:
        pred_text = unique_predictions[pred_idx]
        truth_text = unique_ground_truth[truth_idx]
        
        result["true_positives"].append({
            "prediction": pred_text,
            "ground_truth": truth_text,
            "similarity": similarity
        })
        
        matched_pred_indices.add(pred_idx)
        matched_truth_indices.add(truth_idx)
    
    # Identify false positives and negatives
    false_positive_indices = set(range(len(unique_predictions))) - matched_pred_indices
    false_negative_indices = set(range(len(unique_ground_truth))) - matched_truth_indices
    
    result["false_positives"] = [unique_predictions[i] for i in false_positive_indices]
    result["false_negatives"] = [unique_ground_truth[i] for i in false_negative_indices]
    
    # Calculate metrics
    result["tp_count"] = len(matched_pred_indices)
    result["fp_count"] = len(false_positive_indices)
    result["fn_count"] = len(false_negative_indices)
    
    # Calculate weighted true positives based on similarity scores
    weighted_tp = sum(similarity / 100.0 for _, _, similarity in matches)
    
    # Calculate precision and recall
    if unique_predictions:
        result["precision"] = weighted_tp / len(unique_predictions)
    if unique_ground_truth:
        result["recall"] = weighted_tp / len(unique_ground_truth)
    
    # Calculate F1 score
    if result["precision"] + result["recall"] > 0:
        result["f1_score"] = 2 * (result["precision"] * result["recall"]) / (result["precision"] + result["recall"])
    
    return result

def evaluate_all_cases(method_phenotypes: Dict[str, List[str]], ground_truth_phenotypes: Dict[str, List[str]],
                      similarity_threshold: float = 50.0) -> Dict[str, Dict]:
    results = {}
    
    # Get unique case IDs that have ground truth
    case_ids = set(ground_truth_phenotypes.keys())
    
    for case_id in case_ids:
        # Get phenotypes for this case
        method_case_phenotypes = method_phenotypes.get(case_id, [])
        ground_truth_case_phenotypes = ground_truth_phenotypes[case_id]
        
        # Evaluate this case
        results[case_id] = evaluate_phenotypes(
            method_case_phenotypes, ground_truth_case_phenotypes, similarity_threshold
        )
    
    return results

def calculate_overall_metrics(results: Dict[str, Dict]) -> Dict:
    # Initialize counters
    total_tp = 0
    total_fp = 0
    total_fn = 0
    weighted_tp_sum = 0
    
    # Aggregate counts
    for case_result in results.values():
        total_tp += case_result["tp_count"]
        total_fp += case_result["fp_count"]
        total_fn += case_result["fn_count"]
        
        # Calculate weighted true positives
        for tp in case_result["true_positives"]:
            weighted_tp_sum += tp["similarity"] / 100.0
    
    # Calculate metrics
    precision = weighted_tp_sum / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = weighted_tp_sum / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "tp_count": total_tp,
        "fp_count": total_fp,
        "fn_count": total_fn,
        "weighted_tp": weighted_tp_sum
    }

# Set similarity threshold
similarity_threshold = 50.0

# Evaluate each method
method1_results = evaluate_all_cases(method1_phenotypes, ground_truth_phenotypes, similarity_threshold)
method2_results = evaluate_all_cases(method2_phenotypes, ground_truth_phenotypes, similarity_threshold)

print(f"Evaluated {len(method1_results)} cases for {method1_name}")
print(f"Evaluated {len(method2_results)} cases for {method2_name}")

# def calculate_overall_metrics(results: Dict[str, Dict]) -> Dict:
#     """Calculate overall metrics across all cases."""
#     # Initialize counters
#     total_tp = 0
#     total_fp = 0
#     total_fn = 0
#     weighted_tp_sum = 0
    
#     # Aggregate counts
#     for case_id, case_result in results.items():
#         total_tp += case_result["tp_count"]
#         total_fp += case_result["fp_count"]
#         total_fn += case_result["fn_count"]
        
#         # Calculate weighted true positives
#         for tp in case_result["true_positives"]:
#             weighted_tp_sum += tp["similarity"] / 100.0
    
#     # Calculate metrics
#     precision = weighted_tp_sum / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
#     recall = weighted_tp_sum / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
#     f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
#     return {
#         "precision": precision,
#         "recall": recall,
#         "f1_score": f1_score,
#         "tp_count": total_tp,
#         "fp_count": total_fp,
#         "fn_count": total_fn,
#         "weighted_tp": weighted_tp_sum
#     }

# Calculate overall metrics
method1_overall = calculate_overall_metrics(method1_results)
method2_overall = calculate_overall_metrics(method2_results)

# Create a comparison DataFrame
overall_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1 Score", "True Positives", "False Positives", "False Negatives"],
    method1_name: [
        method1_overall["precision"],
        method1_overall["recall"],
        method1_overall["f1_score"],
        method1_overall["tp_count"],
        method1_overall["fp_count"],
        method1_overall["fn_count"]
    ],
    method2_name: [
        method2_overall["precision"],
        method2_overall["recall"],
        method2_overall["f1_score"],
        method2_overall["tp_count"],
        method2_overall["fp_count"],
        method2_overall["fn_count"]
    ]
})

# Display overall metrics
print("\nOverall Verification Performance Metrics:")
print(overall_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# Plot comparison of precision, recall, and F1 score
metrics_to_plot = overall_df.iloc[:3].copy()
metrics_to_plot = pd.melt(metrics_to_plot, id_vars=["Metric"], var_name="Method", value_name="Score")

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x="Metric", y="Score", hue="Method", data=metrics_to_plot)
plt.title("Overall Verification Performance Comparison")
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on bars
for p in bar_plot.patches:
    bar_plot.annotate(f"{p.get_height():.3f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha = 'center', va = 'bottom', xytext = (0, 5),
                     textcoords = 'offset points')

plt.tight_layout()
plt.show()

def identify_diverging_cases(method1_results: Dict[str, Dict], method2_results: Dict[str, Dict]) -> pd.DataFrame:
    """Identify cases where methods perform differently."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_phenotypes.keys())
    
    # Calculate performance difference for each case
    case_data = []
    for case_id in common_cases:
        # Skip cases without ground truth
        if not ground_truth_phenotypes.get(case_id, []):
            continue
            
        m1_result = method1_results[case_id]
        m2_result = method2_results[case_id]
        
        # Calculate absolute differences
        f1_diff = abs(m1_result["f1_score"] - m2_result["f1_score"])
        precision_diff = abs(m1_result["precision"] - m2_result["precision"])
        recall_diff = abs(m1_result["recall"] - m2_result["recall"])
        
        # Determine which method is better for this case
        better_method = method1_name if m1_result["f1_score"] > m2_result["f1_score"] else method2_name
        
        # Count of phenotypes
        ground_truth_count = len(ground_truth_phenotypes[case_id])
        m1_count = len(method1_phenotypes.get(case_id, []))
        m2_count = len(method2_phenotypes.get(case_id, []))
        
        case_data.append({
            "case_id": case_id,
            f"{method1_name} F1": m1_result["f1_score"],
            f"{method2_name} F1": m2_result["f1_score"],
            "F1 Difference": f1_diff,
            f"{method1_name} Precision": m1_result["precision"],
            f"{method2_name} Precision": m2_result["precision"],
            "Precision Difference": precision_diff,
            f"{method1_name} Recall": m1_result["recall"],
            f"{method2_name} Recall": m2_result["recall"],
            "Recall Difference": recall_diff,
            "Better Method": better_method,
            "Ground Truth Count": ground_truth_count,
            f"{method1_name} Count": m1_count,
            f"{method2_name} Count": m2_count,
        })
    
    # Create DataFrame and sort by F1 difference
    divergence_df = pd.DataFrame(case_data)
    divergence_df = divergence_df.sort_values("F1 Difference", ascending=False)
    
    return divergence_df

# Identify diverging cases
divergence_df = identify_diverging_cases(method1_results, method2_results)

# Display top 10 diverging cases
print("\nTop 10 Cases with Largest Performance Differences:")
print(divergence_df.head(10).to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# Plot F1 score differences for top 10 cases
top_cases = divergence_df.head(10)

plt.figure(figsize=(12, 6))
ind = np.arange(len(top_cases))
width = 0.35

bars1 = plt.bar(ind - width/2, top_cases[f"{method1_name} F1"], width, label=method1_name)
bars2 = plt.bar(ind + width/2, top_cases[f"{method2_name} F1"], width, label=method2_name)

plt.ylabel('F1 Score')
plt.title('F1 Scores for Top Diverging Cases')
plt.xticks(ind, top_cases['case_id'], rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

def analyze_phenotype_sets(case_id: str) -> Dict:
    """Analyze phenotype sets for a specific case."""
    # Get phenotypes for this case
    ground_truth = set(ground_truth_phenotypes.get(case_id, []))
    method1_set = set(method1_phenotypes.get(case_id, []))
    method2_set = set(method2_phenotypes.get(case_id, []))
    
    # Get true positives for each method (using fuzzy matching results)
    if case_id in method1_results:
        method1_tps = {tp["prediction"] for tp in method1_results[case_id]["true_positives"]}
    else:
        method1_tps = set()
        
    if case_id in method2_results:
        method2_tps = {tp["prediction"] for tp in method2_results[case_id]["true_positives"]}
    else:
        method2_tps = set()
    
    # Analyze set differences
    return {
        "case_id": case_id,
        "ground_truth": ground_truth,
        f"{method1_name}_phenotypes": method1_set,
        f"{method2_name}_phenotypes": method2_set,
        f"{method1_name}_only_correct": method1_tps - method2_tps,  # Correct in method1 but not method2
        f"{method2_name}_only_correct": method2_tps - method1_tps,  # Correct in method2 but not method1
        "both_correct": method1_tps & method2_tps,  # Correct in both methods
        "missed_by_both": ground_truth - {gt for tp in method1_results.get(case_id, {}).get("true_positives", []) 
                                       for gt in [tp.get("ground_truth")]} -
                          {gt for tp in method2_results.get(case_id, {}).get("true_positives", []) 
                                       for gt in [tp.get("ground_truth")]},  # Missed by both methods
    }

# Analyze top 5 diverging cases
top_5_case_ids = divergence_df.head(5)["case_id"].tolist()
top_5_analyses = [analyze_phenotype_sets(case_id) for case_id in top_5_case_ids]

# Print analysis for each case
for i, analysis in enumerate(top_5_analyses):
    case_id = analysis["case_id"]
    
    print(f"\n==== Case {case_id} (Rank {i+1}) ====")
    
    # Get F1 scores for this case
    method1_f1 = method1_results.get(case_id, {}).get("f1_score", 0)
    method2_f1 = method2_results.get(case_id, {}).get("f1_score", 0)
    
    # Display scores
    print(f"{method1_name} F1: {method1_f1:.4f}, {method2_name} F1: {method2_f1:.4f}, Difference: {abs(method1_f1 - method2_f1):.4f}")
    
    # Create sets for Venn diagram
    set1 = analysis[f"{method1_name}_phenotypes"]
    set2 = analysis[f"{method2_name}_phenotypes"]
    set3 = analysis["ground_truth"]
    
    # Plot Venn diagram
    plt.figure(figsize=(10, 6))
    venn3([set1, set2, set3], 
          (method1_name, method2_name, "Ground Truth"))
    plt.title(f"Phenotype Set Comparison for Case {case_id}")
    plt.tight_layout()
    plt.show()
    
    # Display phenotype details
    print(f"\nPhenotype Details for Case {case_id}:")
    
    print(f"\nPhenotypes correct only in {method1_name}:")
    for phenotype in analysis[f"{method1_name}_only_correct"]:
        print(f"  - {phenotype}")
    
    print(f"\nPhenotypes correct only in {method2_name}:")
    for phenotype in analysis[f"{method2_name}_only_correct"]:
        print(f"  - {phenotype}")
    
    print("\nPhenotypes correct in both methods:")
    for phenotype in analysis["both_correct"]:
        print(f"  - {phenotype}")
    
    print("\nPhenotypes missed by both methods:")
    for phenotype in analysis["missed_by_both"]:
        print(f"  - {phenotype}")
    
    # Show original text if available
    if case_id in method1_data and "original_text" in method1_data[case_id]:
        clinical_text = method1_data[case_id]["original_text"]
        print(f"\nOriginal Clinical Text (truncated):")
        print(clinical_text[:500] + "..." if len(clinical_text) > 500 else clinical_text)
    elif case_id in method2_data and "original_text" in method2_data[case_id]:
        clinical_text = method2_data[case_id]["original_text"]
        print(f"\nOriginal Clinical Text (truncated):")
        print(clinical_text[:500] + "..." if len(clinical_text) > 500 else clinical_text)
    
    # Show verification statistics if available
    if case_id in method1_data and "stats" in method1_data[case_id]:
        stats1 = method1_data[case_id]["stats"]
        stats2 = method2_data.get(case_id, {}).get("stats", {})
        
        print(f"\nVerification Statistics:")
        print(f"  {method1_name}:")
        for key, value in stats1.items():
            print(f"    - {key}: {value}")
        
        print(f"  {method2_name}:")
        for key, value in stats2.items():
            print(f"    - {key}: {value}")
    
    print("\n" + "-"*80)

def analyze_method_strengths(method1_results, method2_results):
    """Analyze strengths and weaknesses of each method across all cases."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys())
    
    # Initialize sets for strengths analysis
    method1_only_correct = set()
    method2_only_correct = set()
    both_correct = set()
    missed_by_both = set()
    
    # Collect phenotypes across all cases
    for case_id in common_cases:
        if case_id not in ground_truth_phenotypes:
            continue
            
        # Get true positives for each method
        method1_tps = {tp["prediction"] for tp in method1_results[case_id]["true_positives"]}
        method2_tps = {tp["prediction"] for tp in method2_results[case_id]["true_positives"]}
        
        # Update sets
        method1_only_correct.update(method1_tps - method2_tps)
        method2_only_correct.update(method2_tps - method1_tps)
        both_correct.update(method1_tps & method2_tps)
        
        # Get ground truth items missed by both methods
        method1_gt_matches = {tp["ground_truth"] for tp in method1_results[case_id]["true_positives"]}
        method2_gt_matches = {tp["ground_truth"] for tp in method2_results[case_id]["true_positives"]}
        missed = set(ground_truth_phenotypes[case_id]) - method1_gt_matches - method2_gt_matches
        missed_by_both.update(missed)
    
    # Count method performances by case
    method1_better_count = 0
    method2_better_count = 0
    tie_count = 0
    
    for case_id in common_cases:
        if case_id not in ground_truth_phenotypes:
            continue
            
        m1_f1 = method1_results[case_id]["f1_score"]
        m2_f1 = method2_results[case_id]["f1_score"]
        
        if abs(m1_f1 - m2_f1) < 0.01:  # Consider as tie if difference is very small
            tie_count += 1
        elif m1_f1 > m2_f1:
            method1_better_count += 1
        else:
            method2_better_count += 1
    
    # Create summary DataFrame
    summary_df = pd.DataFrame({
        "Category": [f"Phenotypes found only by {method1_name}", 
                    f"Phenotypes found only by {method2_name}",
                    "Phenotypes found by both methods",
                    "Phenotypes missed by both methods",
                    f"Cases where {method1_name} performed better",
                    f"Cases where {method2_name} performed better",
                    "Cases with similar performance"],
        "Count": [len(method1_only_correct),
                len(method2_only_correct),
                len(both_correct),
                len(missed_by_both),
                method1_better_count,
                method2_better_count,
                tie_count]
    })
    
    # Return sets and summary
    return {
        "summary": summary_df,
        "method1_only": method1_only_correct,
        "method2_only": method2_only_correct,
        "both": both_correct,
        "missed": missed_by_both
    }

# Analyze strengths
strengths_analysis = analyze_method_strengths(method1_results, method2_results)

# Display summary
print("\nOverall Method Comparison:")
print(strengths_analysis["summary"].to_string(index=False))

# Plot summary
plt.figure(figsize=(12, 6))
entity_data = strengths_analysis["summary"].iloc[:4].copy()
sns.barplot(x="Category", y="Count", data=entity_data)
plt.title("Phenotype Detection Comparison")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
case_data = strengths_analysis["summary"].iloc[4:].copy()
sns.barplot(x="Category", y="Count", data=case_data)
plt.title("Case Performance Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Display most common unique phenotypes found by each method
print("\nMost Common Phenotypes Found Only by Each Method:")

# Get most common phenotypes unique to each method
method1_common = Counter(list(strengths_analysis["method1_only"])).most_common(15)
method2_common = Counter(list(strengths_analysis["method2_only"])).most_common(15)

# Create DataFrames for display
method1_common_df = pd.DataFrame(method1_common, columns=[f"Phenotype unique to {method1_name}", "Count"])
method2_common_df = pd.DataFrame(method2_common, columns=[f"Phenotype unique to {method2_name}", "Count"])

# Display side by side
print(pd.concat([method1_common_df, method2_common_df], axis=1).to_string(index=False))

# Now let's analyze the verification statistics
def analyze_verification_stats():
    """Analyze verification statistics from step2_verify output."""
    # Initialize counters
    method1_stats = {
        "total_original_entities": 0,
        "total_filtered_hallucinations": 0,
        "total_entities_verified": 0,
        "total_phenotypes_found": 0
    }
    
    method2_stats = {
        "total_original_entities": 0,
        "total_filtered_hallucinations": 0,
        "total_entities_verified": 0,
        "total_phenotypes_found": 0
    }
    
    # Collect stats for each method
    for case_id, case_data in method1_data.items():
        if "stats" in case_data:
            stats = case_data["stats"]
            method1_stats["total_original_entities"] += stats.get("original_entity_count", 0)
            method1_stats["total_filtered_hallucinations"] += stats.get("filtered_hallucinations", 0)
            method1_stats["total_entities_verified"] += stats.get("entities_verified", 0)
            method1_stats["total_phenotypes_found"] += stats.get("phenotypes_found", 0)
    
    for case_id, case_data in method2_data.items():
        if "stats" in case_data:
            stats = case_data["stats"]
            method2_stats["total_original_entities"] += stats.get("original_entity_count", 0)
            method2_stats["total_filtered_hallucinations"] += stats.get("filtered_hallucinations", 0)
            method2_stats["total_entities_verified"] += stats.get("entities_verified", 0)
            method2_stats["total_phenotypes_found"] += stats.get("phenotypes_found", 0)
    
    # Calculate rates
    method1_stats["hallucination_rate"] = (method1_stats["total_filtered_hallucinations"] / method1_stats["total_original_entities"] * 100 
                                         if method1_stats["total_original_entities"] > 0 else 0)
    
    method1_stats["verification_rate"] = (method1_stats["total_phenotypes_found"] / method1_stats["total_entities_verified"] * 100 
                                        if method1_stats["total_entities_verified"] > 0 else 0)
    
    method1_stats["overall_yield"] = (method1_stats["total_phenotypes_found"] / method1_stats["total_original_entities"] * 100 
                                    if method1_stats["total_original_entities"] > 0 else 0)
    
    method2_stats["hallucination_rate"] = (method2_stats["total_filtered_hallucinations"] / method2_stats["total_original_entities"] * 100 
                                         if method2_stats["total_original_entities"] > 0 else 0)
    
    method2_stats["verification_rate"] = (method2_stats["total_phenotypes_found"] / method2_stats["total_entities_verified"] * 100 
                                        if method2_stats["total_entities_verified"] > 0 else 0)
    
    method2_stats["overall_yield"] = (method2_stats["total_phenotypes_found"] / method2_stats["total_original_entities"] * 100 
                                    if method2_stats["total_original_entities"] > 0 else 0)
    
    # Create comparison DataFrame
    stats_df = pd.DataFrame({
        "Metric": ["Original Entities", "Filtered Hallucinations", "Entities Verified", "Phenotypes Found", 
                  "Hallucination Rate (%)", "Verification Rate (%)", "Overall Yield (%)"],
        method1_name: [
            method1_stats["total_original_entities"],
            method1_stats["total_filtered_hallucinations"],
            method1_stats["total_entities_verified"],
            method1_stats["total_phenotypes_found"],
            method1_stats["hallucination_rate"],
            method1_stats["verification_rate"],
            method1_stats["overall_yield"]
        ],
        method2_name: [
            method2_stats["total_original_entities"],
            method2_stats["total_filtered_hallucinations"],
            method2_stats["total_entities_verified"],
            method2_stats["total_phenotypes_found"],
            method2_stats["hallucination_rate"],
            method2_stats["verification_rate"],
            method2_stats["overall_yield"]
        ]
    })
    
    return stats_df

# Analyze verification statistics
verification_stats_df = analyze_verification_stats()

# Display verification statistics
print("\nVerification Statistics Comparison:")
print(verification_stats_df.to_string(index=False, float_format=lambda x: f"{x:.2f}" if isinstance(x, float) else str(x)))

# Plot verification rates
plt.figure(figsize=(10, 6))
rates_df = verification_stats_df.iloc[4:].copy()
rates_df = pd.melt(rates_df, id_vars=["Metric"], var_name="Method", value_name="Percentage")

sns.barplot(x="Metric", y="Percentage", hue="Method", data=rates_df)
plt.title("Verification Process Metrics")
plt.ylabel("Percentage (%)")
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("""
Conclusion:

Based on the analysis above, we can draw the following conclusions about the two verification methods:

1. Overall Performance:
   - [Describe which method performs better overall and why]

2. Strengths of Verification Method A:
   - [List specific types of phenotypes or cases where Method A excels]

3. Strengths of Verification Method B:
   - [List specific types of phenotypes or cases where Method B excels]

4. Verification Process Efficiency:
   - [Compare hallucination filtering and verification rates]

5. Recommendations:
   - [Based on the analysis, provide recommendations for when to use each method or how to combine them]

This analysis provides insights into how the two verification methods compare in terms of phenotype identification accuracy and process efficiency, which can guide decisions about which method to use in different scenarios.
""")

In [ ]:
def analyze_missed_phenotypes(method1_results, method2_results):
    """Analyze phenotypes missed by both verification methods across all cases."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_phenotypes.keys())
    
    # Initialize collections for missed phenotypes
    all_missed = []
    missed_by_case = {}
    
    # Analyze missed phenotypes for each case
    for case_id in common_cases:
        # Get ground truth phenotypes
        gt_phenotypes = ground_truth_phenotypes.get(case_id, [])
        if not gt_phenotypes:
            continue
            
        # Get matched ground truth phenotypes for each method
        method1_gt_matches = {tp["ground_truth"] for tp in method1_results[case_id]["true_positives"]}
        method2_gt_matches = {tp["ground_truth"] for tp in method2_results[case_id]["true_positives"]}
        
        # Find phenotypes missed by both methods
        missed = set(gt_phenotypes) - method1_gt_matches - method2_gt_matches
        
        # Add to collections
        if missed:
            missed_by_case[case_id] = list(missed)
            all_missed.extend(missed)
    
    # Count occurrences of each missed phenotype
    missed_counts = Counter(all_missed)
    
    return {
        "missed_by_case": missed_by_case,
        "most_common_missed": missed_counts.most_common(30),  # Show top 30 most commonly missed
        "total_missed_unique": len(missed_counts),
        "total_missed_occurrences": sum(missed_counts.values())
    }

# Function to analyze phenotype characteristics
def analyze_phenotype_characteristics(phenotypes):
    """Analyze characteristics of phenotypes such as length, complexity, etc."""
    word_counts = [len(p.split()) for p in phenotypes]
    char_counts = [len(p) for p in phenotypes]
    
    return {
        "total_phenotypes": len(phenotypes),
        "avg_word_count": sum(word_counts) / len(word_counts) if word_counts else 0,
        "avg_char_count": sum(char_counts) / len(char_counts) if char_counts else 0,
        "single_word_count": word_counts.count(1),
        "single_word_percent": word_counts.count(1) / len(word_counts) * 100 if word_counts else 0,
        "word_counts": word_counts
    }

# Add extra analysis for comparison between verification methods
def compare_method_errors():
    """Compare error patterns between verification methods."""
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_phenotypes.keys())
    
    # Initialize counters
    error_patterns = {
        "both_correct": 0,
        "method1_only_correct": 0,
        "method2_only_correct": 0,
        "both_wrong": 0,
        "total_phenotypes": 0
    }
    
    # Analyze each case
    for case_id in common_cases:
        gt_phenotypes = ground_truth_phenotypes.get(case_id, [])
        if not gt_phenotypes:
            continue
            
        # Get matched ground truth phenotypes for each method
        method1_matches = {tp["ground_truth"] for tp in method1_results[case_id]["true_positives"]}
        method2_matches = {tp["ground_truth"] for tp in method2_results[case_id]["true_positives"]}
        
        # Compare on each ground truth phenotype
        for phenotype in gt_phenotypes:
            error_patterns["total_phenotypes"] += 1
            
            m1_correct = phenotype in method1_matches
            m2_correct = phenotype in method2_matches
            
            if m1_correct and m2_correct:
                error_patterns["both_correct"] += 1
            elif m1_correct and not m2_correct:
                error_patterns["method1_only_correct"] += 1
            elif not m1_correct and m2_correct:
                error_patterns["method2_only_correct"] += 1
            else:
                error_patterns["both_wrong"] += 1
    
    return error_patterns

# Run the missed phenotypes analysis
print("\n===== DETAILED ANALYSIS OF MISSED PHENOTYPES =====")
missed_analysis = analyze_missed_phenotypes(method1_results, method2_results)

# Additional comparison of error patterns
error_patterns = compare_method_errors()
print("\nComparison of Verification Method Performance:")
print(f"Total ground truth phenotypes: {error_patterns['total_phenotypes']}")
print(f"Both methods correct: {error_patterns['both_correct']} ({error_patterns['both_correct']/error_patterns['total_phenotypes']*100:.1f}%)")
print(f"Only {method1_name} correct: {error_patterns['method1_only_correct']} ({error_patterns['method1_only_correct']/error_patterns['total_phenotypes']*100:.1f}%)")
print(f"Only {method2_name} correct: {error_patterns['method2_only_correct']} ({error_patterns['method2_only_correct']/error_patterns['total_phenotypes']*100:.1f}%)")
print(f"Both methods wrong: {error_patterns['both_wrong']} ({error_patterns['both_wrong']/error_patterns['total_phenotypes']*100:.1f}%)")

# Create a pie chart of error patterns
plt.figure(figsize=(10, 7))
labels = [
    f"Both methods correct\n({error_patterns['both_correct']/error_patterns['total_phenotypes']*100:.1f}%)",
    f"Only {method1_name} correct\n({error_patterns['method1_only_correct']/error_patterns['total_phenotypes']*100:.1f}%)",
    f"Only {method2_name} correct\n({error_patterns['method2_only_correct']/error_patterns['total_phenotypes']*100:.1f}%)",
    f"Both methods wrong\n({error_patterns['both_wrong']/error_patterns['total_phenotypes']*100:.1f}%)"
]
sizes = [
    error_patterns['both_correct'],
    error_patterns['method1_only_correct'],
    error_patterns['method2_only_correct'],
    error_patterns['both_wrong']
]
colors = ['#4CAF50', '#2196F3', '#FFC107', '#F44336']
explode = (0.1, 0, 0, 0.1)  # explode 1st and 4th slice for emphasis

plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
plt.title('Verification Method Error Patterns')
plt.tight_layout()
plt.show()

# Display overall statistics
print(f"\nTotal unique phenotypes missed by both verification methods: {missed_analysis['total_missed_unique']}")
print(f"Total occurrences of missed phenotypes: {missed_analysis['total_missed_occurrences']}")

# Display most commonly missed phenotypes
print("\nTop 30 Most Commonly Missed Phenotypes:")
for phenotype, count in missed_analysis['most_common_missed']:
    print(f"  - '{phenotype}' (missed in {count} cases)")

# Display cases with the most missed phenotypes
cases_by_missed_count = sorted(
    [(case_id, len(entities)) for case_id, entities in missed_analysis['missed_by_case'].items()],
    key=lambda x: x[1], reverse=True
)

print("\nTop 10 Cases with Most Missed Phenotypes:")
for i, (case_id, count) in enumerate(cases_by_missed_count[:10]):
    print(f"\nCase {case_id}: {count} missed phenotypes")
    
    # Show all missed phenotypes for these top cases
    missed_phenotypes = missed_analysis['missed_by_case'][case_id]
    for phenotype in missed_phenotypes:
        print(f"  - '{phenotype}'")
    
    # Show original text if available
    if case_id in method1_data and "original_text" in method1_data[case_id]:
        clinical_text = method1_data[case_id]["original_text"]
        print(f"\nExcerpt from Clinical Text:")
        print(clinical_text[:300] + "..." if len(clinical_text) > 300 else clinical_text)
    elif case_id in method2_data and "original_text" in method2_data[case_id]:
        clinical_text = method2_data[case_id]["original_text"]
        print(f"\nExcerpt from Clinical Text:")
        print(clinical_text[:300] + "..." if len(clinical_text) > 300 else clinical_text)
    
    print("-"*80)

# Analyze characteristics of missed phenotypes
all_missed_phenotypes = [p for case_phenotypes in missed_analysis['missed_by_case'].values() for p in case_phenotypes]
missed_characteristics = analyze_phenotype_characteristics(all_missed_phenotypes)

print("\nCharacteristics of Missed Phenotypes:")
print(f"Average word count: {missed_characteristics['avg_word_count']:.2f}")
print(f"Average character count: {missed_characteristics['avg_char_count']:.2f}")
print(f"Single-word phenotypes: {missed_characteristics['single_word_count']} ({missed_characteristics['single_word_percent']:.1f}%)")

# Compare characteristics of missed phenotypes vs. found phenotypes
found_by_either = []
# Get common case IDs
common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_phenotypes.keys())

for case_id in common_cases:
    if case_id not in ground_truth_phenotypes:
        continue
        
    # Get true positives for each method
    method1_tps = {tp["ground_truth"] for tp in method1_results[case_id]["true_positives"]}
    method2_tps = {tp["ground_truth"] for tp in method2_results[case_id]["true_positives"]}
    
    # Combined found phenotypes
    found_by_either.extend(list(method1_tps | method2_tps))

found_characteristics = analyze_phenotype_characteristics(found_by_either)

print("\nComparison of Found vs. Missed Phenotypes:")
print(f"Found: {found_characteristics['total_phenotypes']} phenotypes, Avg words: {found_characteristics['avg_word_count']:.2f}, Avg chars: {found_characteristics['avg_char_count']:.2f}")
print(f"Missed: {missed_characteristics['total_phenotypes']} phenotypes, Avg words: {missed_characteristics['avg_word_count']:.2f}, Avg chars: {missed_characteristics['avg_char_count']:.2f}")

# Create a visualization of word count distribution
plt.figure(figsize=(12, 6))
# Handle empty lists case
if found_characteristics["word_counts"] and missed_characteristics["word_counts"]:
    max_words = max(max(found_characteristics["word_counts"]), max(missed_characteristics["word_counts"]))
    plt.hist(
        [found_characteristics["word_counts"], missed_characteristics["word_counts"]], 
        bins=range(1, max_words + 2), 
        alpha=0.7,
        label=["Found Phenotypes", "Missed Phenotypes"]
    )
    plt.xticks(range(1, max_words + 1))
else:
    plt.text(0.5, 0.5, "Insufficient data for word count histogram", 
             horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

plt.title("Distribution of Word Counts in Found vs. Missed Phenotypes")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Analyze verification confidence for missed vs. found phenotypes
def analyze_verification_confidence():
    """Analyze verification confidence scores for correct vs. missed phenotypes."""
    method1_confidences = {"correct": [], "incorrect": []}
    method2_confidences = {"correct": [], "incorrect": []}
    
    # Get common case IDs
    common_cases = set(method1_results.keys()) & set(method2_results.keys()) & set(ground_truth_phenotypes.keys())
    
    for case_id in common_cases:
        if case_id not in ground_truth_phenotypes:
            continue
            
        # Get ground truth phenotypes
        gt_phenotypes = set(ground_truth_phenotypes[case_id])
        
        # Process method 1
        if case_id in method1_data and "verified_phenotypes" in method1_data[case_id]:
            for verified in method1_data[case_id]["verified_phenotypes"]:
                if "confidence" in verified and "phenotype" in verified:
                    phenotype = verified["phenotype"]
                    confidence = verified["confidence"] if isinstance(verified["confidence"], (int, float)) else 0
                    
                    # Determine if this is a true positive (with fuzzy matching)
                    is_correct = False
                    for tp in method1_results[case_id]["true_positives"]:
                        if tp["prediction"] == phenotype:
                            is_correct = True
                            break
                            
                    if is_correct:
                        method1_confidences["correct"].append(confidence)
                    else:
                        method1_confidences["incorrect"].append(confidence)
        
        # Process method 2
        if case_id in method2_data and "verified_phenotypes" in method2_data[case_id]:
            for verified in method2_data[case_id]["verified_phenotypes"]:
                if "confidence" in verified and "phenotype" in verified:
                    phenotype = verified["phenotype"]
                    confidence = verified["confidence"] if isinstance(verified["confidence"], (int, float)) else 0
                    
                    # Determine if this is a true positive (with fuzzy matching)
                    is_correct = False
                    for tp in method2_results[case_id]["true_positives"]:
                        if tp["prediction"] == phenotype:
                            is_correct = True
                            break
                            
                    if is_correct:
                        method2_confidences["correct"].append(confidence)
                    else:
                        method2_confidences["incorrect"].append(confidence)
    
    return {
        method1_name: method1_confidences,
        method2_name: method2_confidences
    }

# Only run confidence analysis if confidence scores are available
has_confidence = False
for case_id, case_data in method1_data.items():
    if isinstance(case_data, dict) and "verified_phenotypes" in case_data:
        for verified in case_data["verified_phenotypes"]:
            if isinstance(verified, dict) and "confidence" in verified:
                has_confidence = True
                break
        if has_confidence:
            break

if has_confidence:
    print("\nAnalyzing verification confidence scores...")
    confidence_analysis = analyze_verification_confidence()
    
    # Display average confidence scores
    for method_name, confidences in confidence_analysis.items():
        correct_avg = sum(confidences["correct"]) / len(confidences["correct"]) if confidences["correct"] else 0
        incorrect_avg = sum(confidences["incorrect"]) / len(confidences["incorrect"]) if confidences["incorrect"] else 0
        
        print(f"\n{method_name} confidence scores:")
        print(f"  Correct phenotypes: {len(confidences['correct'])} with avg confidence {correct_avg:.2f}")
        print(f"  Incorrect phenotypes: {len(confidences['incorrect'])} with avg confidence {incorrect_avg:.2f}")
        
        # Plot confidence distributions
        plt.figure(figsize=(10, 6))
        if confidences["correct"] or confidences["incorrect"]:
            bins = np.linspace(0, 1, 11) if 0 <= max(confidences["correct"] + confidences["incorrect"]) <= 1 else 10
            plt.hist(
                [confidences["correct"], confidences["incorrect"]], 
                bins=bins, 
                alpha=0.7,
                label=["Correct Phenotypes", "Incorrect Phenotypes"]
            )
        else:
            plt.text(0.5, 0.5, "No confidence data available", 
                    horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
        
        plt.title(f"{method_name} Confidence Score Distribution")
        plt.xlabel("Confidence Score")
        plt.ylabel("Frequency")
        plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
else:
    print("\nConfidence scores not available in the data. Skipping confidence analysis.")

# Look for potential areas of improvement based on verification stats
print("\nPotential Areas for Improvement:")

# Check for frequent false positives
print("\nMost Common False Positives:")
m1_false_positives = [fp for case_result in method1_results.values() for fp in case_result["false_positives"]]
m2_false_positives = [fp for case_result in method2_results.values() for fp in case_result["false_positives"]]

if m1_false_positives:
    m1_fp_counter = Counter(m1_false_positives)
    print(f"\n{method1_name} top false positives:")
    for fp, count in m1_fp_counter.most_common(10):
        print(f"  - '{fp}' (occurred {count} times)")

if m2_false_positives:
    m2_fp_counter = Counter(m2_false_positives)
    print(f"\n{method2_name} top false positives:")
    for fp, count in m2_fp_counter.most_common(10):
        print(f"  - '{fp}' (occurred {count} times)")
        
# Compare false positive patterns
common_fps = set(m1_false_positives) & set(m2_false_positives)
only_m1_fps = set(m1_false_positives) - set(m2_false_positives)
only_m2_fps = set(m2_false_positives) - set(m1_false_positives)

print(f"\nCommon false positives (both methods): {len(common_fps)}")
print(f"False positives only in {method1_name}: {len(only_m1_fps)}")
print(f"False positives only in {method2_name}: {len(only_m2_fps)}")

# Look for specific types of errors in missed phenotypes
if all_missed_phenotypes:
    print("\nAnalyzing common text patterns in missed phenotypes...")
    
    # Check for common prefixes/suffixes in missed phenotypes
    prefixes = [p.split()[0].lower() if len(p.split()) > 0 else "" for p in all_missed_phenotypes]
    suffixes = [p.split()[-1].lower() if len(p.split()) > 0 else "" for p in all_missed_phenotypes]
    
    prefix_counts = Counter(prefixes)
    suffix_counts = Counter(suffixes)
    
    print("\nMost common first words in missed phenotypes:")
    for prefix, count in prefix_counts.most_common(5):
        if prefix and count > 1:
            print(f"  - '{prefix}' (appears in {count} missed phenotypes)")
    
    print("\nMost common last words in missed phenotypes:")
    for suffix, count in suffix_counts.most_common(5):
        if suffix and count > 1:
            print(f"  - '{suffix}' (appears in {count} missed phenotypes)")
            
    # Look for special characters or patterns in missed phenotypes
    has_numeric = sum(1 for p in all_missed_phenotypes if any(c.isdigit() for c in p))
    has_parentheses = sum(1 for p in all_missed_phenotypes if "(" in p or ")" in p)
    has_punctuation = sum(1 for p in all_missed_phenotypes if any(c in p for c in ".,;:"))
    
    print("\nSpecial patterns in missed phenotypes:")
    print(f"  - Contains numbers: {has_numeric} phenotypes ({has_numeric/len(all_missed_phenotypes)*100:.1f}%)")
    print(f"  - Contains parentheses: {has_parentheses} phenotypes ({has_parentheses/len(all_missed_phenotypes)*100:.1f}%)")
    print(f"  - Contains punctuation: {has_punctuation} phenotypes ({has_punctuation/len(all_missed_phenotypes)*100:.1f}%)")

# Identify potential verification threshold optimizations
if has_confidence:
    for method_name, confidences in confidence_analysis.items():
        if confidences["correct"] and confidences["incorrect"]:
            # Find potential threshold that would maximize true positives while minimizing false positives
            all_confidences = [(conf, "correct") for conf in confidences["correct"]] + [(conf, "incorrect") for conf in confidences["incorrect"]]
            all_confidences.sort(key=lambda x: x[0])
            
            best_threshold = 0
            best_f1 = 0
            
            # Create a list of unique thresholds to try
            unique_thresholds = sorted(set([c[0] for c in all_confidences]))
            
            for threshold in unique_thresholds:
                tp = sum(1 for c in confidences["correct"] if c >= threshold)
                fp = sum(1 for c in confidences["incorrect"] if c >= threshold)
                fn = sum(1 for c in confidences["correct"] if c < threshold)
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                
                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold
            
            print(f"\nPotential optimal confidence threshold for {method_name}: {best_threshold:.2f}")
            print(f"Maximum F1 score at this threshold: {best_f1:.4f}")
            
            # Plot precision-recall curve at various thresholds
            if len(unique_thresholds) > 1:
                thresholds_to_plot = np.linspace(min(unique_thresholds), max(unique_thresholds), 20)
                precisions = []
                recalls = []
                f1_scores = []
                
                for threshold in thresholds_to_plot:
                    tp = sum(1 for c in confidences["correct"] if c >= threshold)
                    fp = sum(1 for c in confidences["incorrect"] if c >= threshold)
                    fn = sum(1 for c in confidences["correct"] if c < threshold)
                    
                    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                    
                    precisions.append(precision)
                    recalls.append(recall)
                    f1_scores.append(f1)
                
                plt.figure(figsize=(10, 6))
                plt.plot(thresholds_to_plot, precisions, 'b-', label='Precision')
                plt.plot(thresholds_to_plot, recalls, 'g-', label='Recall')
                plt.plot(thresholds_to_plot, f1_scores, 'r-', label='F1 Score')
                plt.axvline(x=best_threshold, color='k', linestyle='--', label=f'Best Threshold: {best_threshold:.2f}')
                plt.title(f"{method_name} - Performance Metrics at Different Confidence Thresholds")
                plt.xlabel("Confidence Threshold")
                plt.ylabel("Score")
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
        else:
            print(f"\nNot enough data to optimize threshold for {method_name}")